# Joint trajectory fit with measured $V_{\rm esc}(t)$

**Goal.** Replace the constant-$V_b$ assumption in the §1.3.6 trajectory model
with the empirical cumulative escape curve $V_{\rm esc}(x)$ measured in §1.2,
and jointly fit the universal coefficients $(C_{d,\rm eff},\,C_{AM})$ together
with the per-shot initial loading $V_b^{(i)}(0)$ across four $U_p=120\ \mathrm{mm/s}$
shots (`test_16, test_24, test_26, test_27`). The vertical EOM is

$$
(1+C_{AM})\,V_{VR}\,\ddot z
   \;=\; g\bigl[V_b^{(i)}(0) - V_{\rm esc}\bigl(x_{\rm ring}^{(i)}(t)\bigr)\bigr]
   \;-\; C_{d,\rm eff}\,V_{VR}^{2/3}\,|\dot z|\dot z,
$$

with $V_{VR}=2\pi^2 R a^2$, $R=23\ \mathrm{mm}$, $a=9\ \mathrm{mm}$ (chapter §1.3.5).

**What's new vs the existing pipeline.**
1. The forward model is an ODE integrated with `solve_ivp`, replacing the
   closed-form $\ln\cosh$ solution that assumed $V_b(t)\equiv V_b$.
2. The §1.2 bubble-counting result enters as a known input curve
   $V_{\rm esc}(x)$, not as a free function fit alongside the trajectory.
3. $V_b^{(i)}(0)$ becomes a per-shot fit parameter instead of being inferred
   from the residual via $V_b^{(i)}(0)=V_{\rm count}^{(i)}+V_b^{(i)}(\infty)$
   only post-hoc.

The bookkeeping (profiled per-shot variance, L-BFGS-B with bounds, baseline
= mean of first 20 cleaned frames, delta-method propagation) follows §1.3.6
unchanged.

## 2. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.optimize import minimize, least_squares

import trajectory_pipeline_v2 as tp
from trajectory_pipeline_v2 import analyse_shot, V_ring, G, model2_lncosh

# Chapter §1.3.5 ring geometry (not the pipeline default a=7).
R_MM = 23.0
A_MM = 9.0
V_VR = V_ring(R_MM, A_MM)
D_RING = 40.0  # mm, ring diameter for x/D normalisation

print(f"R = {R_MM} mm,  a = {A_MM} mm")
print(f"V_VR = 2 pi^2 R a^2 = {V_VR:.2f} mm^3 = {V_VR*1e-3:.4f} mL")
print(f"g    = {G} mm/s^2")
print(f"D_ring = {D_RING} mm")

np.random.seed(0)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 140,
                     'axes.grid': True, 'grid.alpha': 0.3})

## 3. Load escape events per condition

The four conditions registered for the joint fit (matching `section_1_2_figures.ipynb`).
Per-event volume is $V_i = \tfrac{4}{3}\pi r_i^3$ in $\mu\mathrm{L}$ (note $r_i$ is
the bubble radius in mm and the resulting cubic-mm equals $\mu\mathrm{L}$).

In [ ]:
ESCAPE_DIR = Path('out')  # change to '/home/zl483/bubbles/out' on the lab machine
if not ESCAPE_DIR.exists():
    ESCAPE_DIR = Path('/home/zl483/bubbles/out')

# (condition label, escape CSV filename, U_p [mm/s], b, V_nom [uL])
ESCAPE_SHOTS = [
    ('120, b=1', 'bubbles_03.csv', 120, 1, 20),
    ('120, b=3', 'bubbles_01.csv', 120, 3, 40),
    ('200, b=1', 'bubbles_06.csv', 200, 1, 20),
    ('200, b=3', 'bubbles_05.csv', 200, 3, 40),
]

def load_escape(fn):
    df = pd.read_csv(ESCAPE_DIR / fn)
    return df['R_world'].values, df['Y_world'].values, df['X_world'].values

conds = {}  # label -> dict(R, Y, X, V, Up, b, Vnom, files)
for lab, fn, Up, b, Vnom in ESCAPE_SHOTS:
    R, Y, X = load_escape(fn)
    if lab not in conds:
        conds[lab] = dict(R=[], Y=[], X=[], Up=Up, b=b, Vnom=Vnom, files=[])
    conds[lab]['R'].append(R)
    conds[lab]['Y'].append(Y)
    conds[lab]['X'].append(X)
    conds[lab]['files'].append(fn)

for lab, d in conds.items():
    d['R'] = np.concatenate(d['R'])
    d['Y'] = np.concatenate(d['Y'])
    d['X'] = np.concatenate(d['X'])
    d['V'] = 4/3 * np.pi * d['R']**3            # uL per event
    print(f"{lab:<10}  N={len(d['R']):>4}   V_count={d['V'].sum():7.2f} uL"
          f"   V_nom={d['Vnom']:>3} uL  files={d['files']}")

## 4. Build $V_{\rm esc}(x)$ per condition

Sort events by $x$, cumulative-sum the per-event volumes, pad $(0,0)$ at the
left and the plateau value at the right. Linear interpolation; outside the
event support we hold the boundary value (no extrapolation).

In [ ]:
def build_V_esc_x(x_evt, V_evt):
    order = np.argsort(x_evt)
    xs = np.concatenate([[0.0], x_evt[order]])
    cV = np.concatenate([[0.0], np.cumsum(V_evt[order])])
    return interp1d(xs, cV, kind='linear', bounds_error=False,
                    fill_value=(0.0, cV[-1]))

for lab, d in conds.items():
    d['Vesc_x'] = build_V_esc_x(d['Y'], d['V'])
    d['V_esc_inf'] = float(d['Vesc_x'](1e6))

# diagnostic plot
fig, ax = plt.subplots(figsize=(7, 4))
xx = np.linspace(0, 800, 600)
for lab, d in conds.items():
    ax.plot(xx / D_RING, d['Vesc_x'](xx), lw=1.6, label=lab)
ax.set_xlabel(r'$x / D$'); ax.set_ylabel(r'$V_{\rm esc}(x)$ ($\mu$L)')
ax.set_title(r'Cumulative escape volume $V_{\rm esc}(x)$ per condition')
ax.legend(fontsize=8); plt.show()

## 5. Load trajectory data per shot

The four shots used in Chapter Table 1.4 (all at $U_p=120\ \mathrm{mm/s}$):
`test_16, test_24, test_26, test_27`. Edit `TRAJ_DIR` and the V_nom/b
assignments to match the lab convention. If trajectory CSVs are not yet
available, set `SYNTHESISE_TRAJ = True` to fabricate four trajectories from
known $(C_{d,\rm eff},C_{AM},V_b(0))$ values so the rest of the notebook can
run end-to-end as a sanity test (this is the synthetic check described at
the end of the prompt).

In [ ]:
TRAJ_DIR = Path('traj')                       # set to the actual location
SYNTHESISE_TRAJ = not TRAJ_DIR.exists()        # auto-fall-back to synthetic

# (label, csv filename, V_nom [uL], U_p [mm/s], b, condition label)
TRAJ_SHOTS = [
    ('test_23', 'test_23.csv', 40, 120, 3, '120, b=3'),
    ('test_24', 'test_24.csv', 20, 120, 1, '120, b=1'),
    ('test_28', 'test_28.csv', 40, 120, 3, '120, b=3'),
    ('test_30', 'test_30.csv', 20, 120, 1, '120, b=1'),
]

per_shot = []

if SYNTHESISE_TRAJ:
    print("TRAJ_DIR not found -- generating synthetic trajectories.\n"
          "Swap to the real CSVs when ready (or change TRAJ_DIR).")
    # Synthesize trajectories using the same simulate_z used downstream,
    # so the rest of the notebook exercises the real pipeline.
    rng = np.random.default_rng(0)
    TRUE_Cd, TRUE_CAM = 3.0, 0.6
    TRUE_Vb0 = {'test_23': 35.0, 'test_24': 18.0,
                'test_28': 36.0, 'test_30': 19.5}
    SIGMA_NOISE_MM = 2.0

    # Lazy stub of simulate_z used only here (the real one is defined in §8).
    def _stub_simulate(t_eval, V_b0, Cd, CAM, V_esc_t):
        def rhs(t, y):
            z, w = y
            Vb = max(0.0, V_b0 - V_esc_t(t))
            dw = G * Vb / ((1+CAM) * V_VR) \
                 - Cd * V_VR**(-1/3) / (1+CAM) * w * abs(w)
            return [w, dw]
        sol = solve_ivp(rhs, (0, t_eval[-1]), [0.0, 0.0],
                        t_eval=t_eval, method='RK45',
                        rtol=1e-7, atol=1e-7, max_step=0.05)
        return sol.y[0]

    for lab, _, Vnom, Up, b, cond in TRAJ_SHOTS:
        # Realistic ring kinematics: x = U_p t (small decel ignored), 5 s total.
        N = 250
        t = np.linspace(0, 5.0, N)
        x_ring = Up * t                       # downstream position (mm)
        # build V_esc_t for this shot using the chosen condition's Vesc_x
        Vesc_x = conds[cond]['Vesc_x']
        V_esc_t = lambda tq, _x=x_ring, _t=t, _v=Vesc_x: _v(np.interp(tq, _t, _x))
        z_true = _stub_simulate(t, TRUE_Vb0[lab], TRUE_Cd, TRUE_CAM, V_esc_t)
        z_obs = z_true + rng.normal(0, SIGMA_NOISE_MM, N)
        # mimic analyse_shot's result keys we use later
        res = {
            'label': lab,
            't': t, 'x': x_ring, 'z': z_obs,
            't_raw': t, 'x_raw': x_ring, 'z_raw': z_obs,
            'keep_mask': np.ones(N, bool),
            'V_b_uL': Vnom, 'U_p': Up, 'cond': cond, 'b': b, 'V_nom': Vnom,
            'model2': {'params': np.array([z_obs[:20].mean(), 12.0, 0.7]),
                       'z_launch': z_obs[:20].mean(), 'w_inf': 12.0, 'tau': 0.7,
                       'rms': 0.0},
            '_truth': dict(Cd=TRUE_Cd, CAM=TRUE_CAM, Vb0=TRUE_Vb0[lab]),
        }
        per_shot.append(res)
else:
    for lab, fn, Vnom, Up, b, cond in TRAJ_SHOTS:
        res = analyse_shot(TRAJ_DIR / fn, V_b_uL=Vnom, U_p=Up,
                           R_mm=R_MM, a_mm=A_MM, label=lab)
        res['cond'] = cond; res['b'] = b; res['V_nom'] = Vnom
        per_shot.append(res)

for r in per_shot:
    p2 = r['model2']
    print(f"{r['label']:<8}  N={len(r['t']):>4}  cond={r['cond']:<10} "
          f"V_nom={r['V_nom']:>3} uL   ln-cosh:  w_inf={p2['w_inf']:6.2f},"
          f" tau={p2['tau']:5.2f},  rms={p2['rms']:.2f}")

## 6. Compose $V_{\rm esc}(t)$ per shot

Linear interpolant of the measured $x_{\rm ring}(t)$, with linear extrapolation
outside the window using the late-50% slope on the right and the early-30%
slope on the left. Then $V_{\rm esc}^{(i)}(t) = V_{\rm esc}^{(\mathrm{cond}(i))}\!\bigl(x_{\rm ring}^{(i)}(t)\bigr)$.

In [ ]:
def make_x_of_t(t, x):
    t = np.asarray(t); x = np.asarray(x)
    interp = interp1d(t, x, kind='linear', bounds_error=False, fill_value=np.nan)
    n_late = max(2, int(0.5 * len(t)))
    n_early = max(2, int(0.3 * len(t)))
    s_late = np.polyfit(t[-n_late:], x[-n_late:], 1)[0]
    s_early = np.polyfit(t[:n_early], x[:n_early], 1)[0]
    t_lo, t_hi = t[0], t[-1]
    x_lo, x_hi = x[0], x[-1]
    def f(tq):
        scalar = np.isscalar(tq) or (np.ndim(tq) == 0)
        ta = np.atleast_1d(np.asarray(tq, float))
        out = interp(ta)
        below = ta < t_lo
        above = ta > t_hi
        out = np.where(below, x_lo + s_early * (ta - t_lo), out)
        out = np.where(above, x_hi + s_late  * (ta - t_hi), out)
        return float(out[0]) if scalar else out
    return f

for r in per_shot:
    Vesc_x = conds[r['cond']]['Vesc_x']
    r['x_of_t'] = make_x_of_t(r['t'], r['x'])
    def _make_Vesc_t(xt, vx):
        def f(tq):
            return vx(xt(tq))
        return f
    r['V_esc_t'] = _make_Vesc_t(r['x_of_t'], Vesc_x)
    r['V_esc_inf'] = float(Vesc_x(1e6))

# two-panel diagnostic
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
cond_colors = {lab: c for lab, c in zip(conds.keys(), ['C0','C3','C2','C4'])}
tt = np.linspace(0, max(r['t'].max() for r in per_shot), 400)
for r in per_shot:
    c = cond_colors[r['cond']]
    axes[0].plot(tt, r['x_of_t'](tt) / D_RING, color=c, lw=1.3)
    axes[0].scatter(r['t'], r['x']/D_RING, s=3, color=c, alpha=0.4,
                    label=f"{r['label']}  ({r['cond']})")
    axes[1].plot(tt, r['V_esc_t'](tt), color=c, lw=1.3, label=r['label'])
axes[0].set_xlabel('t (s)'); axes[0].set_ylabel(r'$x_{\rm ring}/D$')
axes[0].set_title('Ring downstream travel'); axes[0].legend(fontsize=7)
axes[1].set_xlabel('t (s)'); axes[1].set_ylabel(r'$V_{\rm esc}(t)$ ($\mu$L)')
axes[1].set_title('Cumulative escape sampled along trajectory')
axes[1].legend(fontsize=7)
fig.tight_layout(); plt.show()

## 7. Closed-form moment-matching

A cheap initial guess before NLLS. Two robust observables per shot:

- **Early acceleration.** Quadratic fit to $z(t)$ on $t<0.25\,t_{\max}$ gives
  $A^{(i)} = 2 p_0$, the leading coefficient (since $z\approx \tfrac12 A t^2$).
  At $t=0$ the ODE collapses to
  $A^{(i)} = g\,V_b^{(i)}(0) / \bigl[(1+C_{AM})V_{VR}\bigr]$, pinning $V_b^{(i)}(0)$ given $C_{AM}$.
- **Terminal velocity.** Linear slope on $t>0.5\,t_{\max}$ gives $W_\infty^{(i)}$.
  At $t\to\infty$ the ODE gives
  $W_\infty^{(i)2} = g\,V_b^{(i)}(\infty) / \bigl[C_{d,\rm eff} V_{VR}^{2/3}\bigr]$, pinning
  $V_b^{(i)}(\infty)$ given $C_{d,\rm eff}$.

Mass conservation across the trajectory closes the system:
$V_b^{(i)}(0) - V_b^{(i)}(\infty) = V_{\rm esc}^{(i)}(\infty)$. Substituting,

$$
V_{VR}\,C_{AM} \;-\; \frac{V_{VR}^{2/3}\,(W_\infty^{(i)})^2}{A^{(i)}}\,C_{d,\rm eff}
\;=\; \frac{g\,V_{\rm esc}^{(i)}(\infty)}{A^{(i)}} - V_{VR}.
$$

Stack the four shots and solve by least squares for $(C_{AM},\,C_{d,\rm eff})$.
Back-substitute $V_b^{(i)}(0) = A^{(i)}(1+C_{AM}) V_{VR}/g$.

In [ ]:
rows = []
moments = []
for r in per_shot:
    # Use the baseline ln-cosh fit's (w_inf, tau) directly. At t=0 the
    # ODE collapses to z''(0)=g V_b(0)/[(1+C_AM)V_VR]; the ln-cosh model
    # yields z''(0)=w_inf/tau with z'(0)=0 enforced, which is exactly the
    # acceleration we want.  At t->inf the same model gives dz/dt -> w_inf,
    # matching the terminal-velocity balance.  A free quadratic on a
    # window comparable to tau biases A toward 0 -- avoided here.
    p2 = r['model2']
    A_i = p2['w_inf'] / p2['tau']
    W_i = p2['w_inf']
    moments.append((A_i, W_i, r['V_esc_inf']))
    rows.append({'shot': r['label'], 'cond': r['cond'],
                 'A_mm_s2': A_i, 'W_inf_mm_s': W_i,
                 'V_esc_inf_uL': r['V_esc_inf']})

# Linear system  M x = b, x = (C_AM, C_d_eff)
M = np.array([[V_VR, -V_VR**(2/3) * W**2 / A] for (A, W, _) in moments])
rhs = np.array([G * Ve / A - V_VR for (A, _, Ve) in moments])
(C_AM_mm, C_d_mm), *_ = np.linalg.lstsq(M, rhs, rcond=None)

# Back-substitute V_b(0) per shot
Vb0_mm = [A_i * (1 + C_AM_mm) * V_VR / G for (A_i, _, _) in moments]
for row, vb in zip(rows, Vb0_mm):
    row['V_b0_uL_mm'] = vb
    row['V_b_inf_uL_mm'] = vb - row['V_esc_inf_uL']

df_mm = pd.DataFrame(rows)
print(f"Moment-matching:  C_AM = {C_AM_mm:.3f},  C_d_eff = {C_d_mm:.3f}\n")
print(df_mm.to_string(index=False, float_format=lambda x: f'{x:8.3f}'))

## 8. ODE forward model

`solve_ivp` with `RK45`, `atol=rtol=1e-7`, `max_step=0.05`. Drag uses
$|\dot z|\dot z$ rather than $\dot z^2$ so the sign is correct if $w$ ever
goes negative (it shouldn't, but `least_squares` may probe such regions).

In [ ]:
def simulate_z(t_eval, z_launch, V_b0, C_d_eff, C_AM, V_esc_t, V_VR_=V_VR):
    one_pCAM = 1.0 + C_AM
    coef = C_d_eff * V_VR_**(-1.0/3.0) / one_pCAM
    def rhs(t, y):
        z, w = y
        Vb = V_b0 - V_esc_t(t)
        if Vb < 0.0:
            Vb = 0.0
        return [w, G * Vb / (one_pCAM * V_VR_) - coef * w * abs(w)]
    try:
        sol = solve_ivp(rhs, (t_eval[0], t_eval[-1]), [z_launch, 0.0],
                        t_eval=t_eval, method='RK45',
                        rtol=1e-7, atol=1e-7, max_step=0.1)
        if not sol.success:
            return np.full_like(t_eval, np.nan, dtype=float)
        return sol.y[0]
    except Exception:
        return np.full_like(t_eval, np.nan, dtype=float)

# quick sanity check on shot 0
r0 = per_shot[0]
z_launch0 = r0['z'][:20].mean()
z_sim0 = simulate_z(r0['t'], z_launch0, df_mm['V_b0_uL_mm'].iloc[0],
                    C_d_mm, C_AM_mm, r0['V_esc_t'])
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(r0['t'], r0['z'], s=8, alpha=0.5, label=f"data ({r0['label']})")
ax.plot(r0['t'], z_sim0, 'k-', lw=1.6,
        label='ODE @ moment-matching (Cd, CAM, Vb0)')
ax.set_xlabel('t (s)'); ax.set_ylabel('z (mm)')
ax.legend(); plt.show()

## 9. Joint NLL (profiled per-shot variance)

Same construction as §1.3.6 eq. 1.31: with per-shot Gaussian residuals and
$\sigma_i^2$ profiled out, the negative log-likelihood reduces to

$$
\ell(\theta) \;=\; \sum_{i=1}^{N}\frac{n_i}{2}\log\bigl(\mathrm{RSS}_i(\theta)\bigr) + \mathrm{const}.
$$

`pack` / `unpack` keep $(C_{d,\rm eff},\,C_{AM},\,V_b^{(1)}(0),\dots,V_b^{(N)}(0))$
in a single vector. Bad ODE evals get a hard penalty (RSS clipped at $10^{-12}$ from below).

In [ ]:
N_SHOTS = len(per_shot)
z_launches = np.array([r['z'][:20].mean() for r in per_shot])

def pack(Cd, CAM, V_b0_list):
    return np.concatenate([[Cd, CAM], np.asarray(V_b0_list, float)])

def unpack(theta, N=N_SHOTS):
    return float(theta[0]), float(theta[1]), np.asarray(theta[2:2+N], float)

def per_shot_rss(theta, shots=per_shot, zls=z_launches):
    Cd, CAM, Vb0s = unpack(theta, len(shots))
    out = np.empty(len(shots))
    for i, r in enumerate(shots):
        z_sim = simulate_z(r['t'], zls[i], Vb0s[i], Cd, CAM, r['V_esc_t'])
        if np.any(~np.isfinite(z_sim)):
            out[i] = 1e30
        else:
            out[i] = np.sum((r['z'] - z_sim)**2)
    return out

def neg_log_lik(theta, shots=per_shot, zls=z_launches):
    rss = per_shot_rss(theta, shots, zls)
    rss = np.maximum(rss, 1e-12)
    n_i = np.array([len(r['t']) for r in shots])
    return 0.5 * np.sum(n_i * np.log(rss))

def residual_vector(theta, shots=per_shot, zls=z_launches):
    Cd, CAM, Vb0s = unpack(theta, len(shots))
    parts = []
    for i, r in enumerate(shots):
        z_sim = simulate_z(r['t'], zls[i], Vb0s[i], Cd, CAM, r['V_esc_t'])
        if np.any(~np.isfinite(z_sim)):
            parts.append(np.full_like(r['z'], 1e6))
        else:
            parts.append(r['z'] - z_sim)
    return np.concatenate(parts)

## 10. Joint fit

Initial point: moment-matching $(C_{AM},\,C_{d,\rm eff})$ when both are in bounds,
otherwise the chapter Table 1.4 values $(3.76,\,0.69)$. Per-shot $V_b^{(i)}(0)$
seeded from the baseline ln-cosh acceleration $A=w_\infty/\tau$ plus a fraction
of $V_{\rm esc}^{(i)}(\infty)$.

Bounds: $C_{d,\rm eff}\in[0.5,\,10]$, $C_{AM}\in[0,\,2]$,
$V_b^{(i)}(0)\in[V_{\rm esc}^{(i)}(\infty)+0.5,\,200]\ \mu\mathrm{L}$
(initial inventory must exceed total escape).

**Optimiser**: a single `scipy.optimize.least_squares` call (TRF, with the
bounds). With per-shot $n_i$ within ~10\% of each other, minimising
$\sum r_i^2$ matches the profiled NLL
$\sum (n_i/2)\log\mathrm{RSS}_i$ up to a near-constant; the residual
form is well-scaled, supports bounds natively, and converges in tens of
function evaluations (vs hundreds for L-BFGS-B on the log-RSS surface).
The profiled NLL itself is still evaluated for diagnostics and used by
the Hessian-based uncertainties below.

In [ ]:
V_esc_inf_arr = np.array([r['V_esc_inf'] for r in per_shot])

# Fall back to chapter Table 1.4 values when moment-matching is
# inconsistent (negative or out-of-bounds): the universal combination is
# well-determined but the marginals are not, so a single bad shot can
# wreck the lstsq solution.
if (0.5 < C_d_mm < 10.0) and (0.0 < C_AM_mm < 2.0):
    Cd_init, CAM_init = C_d_mm, C_AM_mm
else:
    print('moment-matching out of bounds -- using chapter (3.76, 0.69) as init')
    Cd_init, CAM_init = 3.76, 0.69

# Per-shot V_b(0) guess: A_baseline (1+C_AM) V_VR / g + 0.5 V_esc_inf.
Vb0_phys = np.array([r['model2']['w_inf'] / r['model2']['tau']
                     * (1 + CAM_init) * V_VR / G
                     + 0.5 * r['V_esc_inf']
                     for r in per_shot])
Vb0_init = np.maximum(Vb0_phys, V_esc_inf_arr + 1.0)
theta0 = pack(Cd_init, CAM_init, Vb0_init)

lb = np.concatenate([[0.5, 0.0], V_esc_inf_arr + 0.5])
ub = np.concatenate([[10.0, 2.0], np.full(N_SHOTS, 200.0)])
theta0 = np.clip(theta0, lb + 1e-6, ub - 1e-6)

print(f'theta0 = {theta0}')
print(f'lb     = {lb}')
print(f'ub     = {ub}\n')

# Single bounded NLS fit. With per-shot sample sizes within ~10% of each
# other, minimising sum r_i^2 matches the profiled NLL
# sum (n_i/2) log(RSS_i) up to a near-constant; the simpler residual form
# is well-scaled and converges in tens of function evaluations.
# diff_step must exceed the ODE solver rtol (1e-7) so the Jacobian is
# signal, not solver noise.
ls = least_squares(residual_vector, theta0, bounds=(lb, ub),
                   diff_step=1e-3,
                   xtol=1e-10, ftol=1e-10, gtol=1e-10,
                   max_nfev=200, verbose=1)
theta_hat = ls.x
Cd_hat, CAM_hat, Vb0_hat = unpack(theta_hat)

# Report NLL for diagnostics (matches what bootstrap CIs will use)
nll_hat = neg_log_lik(theta_hat)

print(f'\nleast_squares: success={ls.success}, message={ls.message!r}')
print(f'                cost={ls.cost:.4f}, nfev={ls.nfev}')
print(f'                NLL ={nll_hat:.4f}')
print(f'\nC_d_eff = {Cd_hat:.4f}')
print(f'C_AM    = {CAM_hat:.4f}')
for r, vb in zip(per_shot, Vb0_hat):
    print(f'  V_b0[{r["label"]}] = {vb:7.3f} uL '
          f'(V_esc_inf = {r["V_esc_inf"]:6.3f}, '
          f'V_b_inf = {vb - r["V_esc_inf"]:6.3f})')


## 11. Hessian-based uncertainties

Central-difference Hessian of $-\log\mathcal L$ at $\hat\theta$; covariance =
$H^{-1}$. Standard errors from $\sqrt{\mathrm{diag}\,\Sigma}$. Marginal
correlation $\rho(C_{d,\rm eff},\,C_{AM})$ from the upper-left $2\times2$.
Delta-method on the well-determined combination $R=(1+C_{AM})/C_{d,\rm eff}$.

In [ ]:
def hess_fd(f, x, h_rel=3e-4):
    n = len(x); H = np.zeros((n, n))
    fx = f(x)
    h = h_rel * (np.abs(x) + 1.0)
    # diagonal
    for i in range(n):
        xp = x.copy(); xp[i] += h[i]
        xm = x.copy(); xm[i] -= h[i]
        H[i, i] = (f(xp) - 2*fx + f(xm)) / h[i]**2
    # off-diagonal
    for i in range(n):
        for j in range(i+1, n):
            xpp = x.copy(); xpp[i] += h[i]; xpp[j] += h[j]
            xpm = x.copy(); xpm[i] += h[i]; xpm[j] -= h[j]
            xmp = x.copy(); xmp[i] -= h[i]; xmp[j] += h[j]
            xmm = x.copy(); xmm[i] -= h[i]; xmm[j] -= h[j]
            H[i, j] = (f(xpp) - f(xpm) - f(xmp) + f(xmm)) / (4 * h[i] * h[j])
            H[j, i] = H[i, j]
    return H

H = hess_fd(neg_log_lik, theta_hat, h_rel=3e-4)
try:
    Sigma = np.linalg.inv(H)
    se = np.sqrt(np.clip(np.diag(Sigma), 0, None))
except np.linalg.LinAlgError:
    Sigma = np.full_like(H, np.nan)
    se = np.full(len(theta_hat), np.nan)

# rho(C_d, C_AM)
s_dd = Sigma[0, 0]; s_aa = Sigma[1, 1]; s_da = Sigma[0, 1]
rho_dCAM = s_da / np.sqrt(s_dd * s_aa) if s_dd > 0 and s_aa > 0 else np.nan

# Delta method on R = (1+CAM)/Cd
gR = np.array([-(1.0 + CAM_hat) / Cd_hat**2, 1.0 / Cd_hat])
varR = float(gR @ Sigma[:2, :2] @ gR)
R_hat = (1.0 + CAM_hat) / Cd_hat
sigR = np.sqrt(max(varR, 0.0))

param_names = ['C_d_eff', 'C_AM'] + [f"V_b0[{r['label']}]" for r in per_shot]
df_unc = pd.DataFrame({'param': param_names,
                       'value': theta_hat, 'std_err': se})
print(df_unc.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))
print(f"\nrho(C_d_eff, C_AM) = {rho_dCAM:+.3f}")
print(f"R = (1+C_AM)/C_d_eff = {R_hat:.4f} ± {sigR:.4f}")

## 12. Comparison vs constant-$V_b$ baseline

Per shot: left = $z(t)$ with data, baseline $\ln\cosh$ fit, and new ODE fit;
right = residuals from both. Table compares per-shot $V_b$ (baseline constant
fit) with $V_b^{(i)}(0)$ and $V_b^{(i)}(\infty) = V_b^{(i)}(0) - V_{\rm esc}^{(i)}(\infty)$
from the new fit, plus RMS for both fits.

In [ ]:
fig, axes = plt.subplots(N_SHOTS, 2, figsize=(11, 2.6 * N_SHOTS),
                          sharex='col')
if N_SHOTS == 1: axes = axes[None, :]
rows_cmp = []
for i, (r, vb0) in enumerate(zip(per_shot, Vb0_hat)):
    t, z = r['t'], r['z']
    z_new = simulate_z(t, z_launches[i], vb0, Cd_hat, CAM_hat, r['V_esc_t'])
    p2 = r['model2']
    z_base = model2_lncosh(t, *p2['params']) if 'params' in p2 else \
             model2_lncosh(t, p2['z_launch'], p2['w_inf'], p2['tau'])
    rms_new = np.sqrt(np.mean((z - z_new)**2))
    rms_base = np.sqrt(np.mean((z - z_base)**2))
    rows_cmp.append({'shot': r['label'],
                     'V_b_baseline_uL': p2.get('w_inf', np.nan)**2 *
                         3.612 * V_VR**(2/3) / G,
                     'V_b0_new_uL': vb0,
                     'V_b_inf_new_uL': vb0 - r['V_esc_inf'],
                     'V_esc_inf_uL': r['V_esc_inf'],
                     'rms_baseline_mm': rms_base,
                     'rms_new_mm': rms_new})

    ax = axes[i, 0]
    ax.scatter(t, z, s=8, alpha=0.4, color='C0', label='data')
    ax.plot(t, z_base, 'C1--', lw=1.3, label=f'baseline (RMS={rms_base:.2f})')
    ax.plot(t, z_new, 'k-', lw=1.6, label=f'ODE (RMS={rms_new:.2f})')
    ax.set_ylabel('z (mm)')
    ax.legend(fontsize=7); ax.set_title(r['label'])

    ax = axes[i, 1]
    ax.scatter(t, z - z_base, s=8, color='C1', alpha=0.5, label='baseline')
    ax.scatter(t, z - z_new, s=8, color='k', alpha=0.7, label='ODE')
    ax.axhline(0, color='k', lw=0.4)
    ax.set_ylabel('residual (mm)'); ax.legend(fontsize=7)

axes[-1, 0].set_xlabel('t (s)'); axes[-1, 1].set_xlabel('t (s)')
fig.tight_layout(); plt.show()

df_cmp = pd.DataFrame(rows_cmp)
print(df_cmp.to_string(index=False, float_format=lambda x: f'{x:9.3f}'))

## 13. Bootstrap over $V_{\rm esc}$

Resample escape events **per condition** with replacement, rebuild
$V_{\rm esc}(x)$, then refit with L-BFGS-B starting from $\hat\theta$ (clipped
into the new bounds). The trajectory data itself is not resampled — only the
escape-curve input. `B=50` by default.

In [ ]:
def refit_with_resampled_Vesc(B=50, seed=1, theta_start=None):
    rng = np.random.default_rng(seed)
    th0 = theta_hat if theta_start is None else theta_start
    draws = []
    for b in range(B):
        # Resample events per condition, rebuild V_esc_x.
        cond_resampled = {}
        for lab, d in conds.items():
            n = len(d['R'])
            idx = rng.integers(0, n, n)
            Vesc_x = build_V_esc_x(d['Y'][idx], d['V'][idx])
            cond_resampled[lab] = dict(Vesc_x=Vesc_x,
                                       V_esc_inf=float(Vesc_x(1e6)))

        # Rebuild per-shot V_esc_t using the original x_of_t.
        shots_b = []
        V_esc_inf_b = np.empty(N_SHOTS)
        for i, r in enumerate(per_shot):
            cr = cond_resampled[r['cond']]
            Vesc_x = cr['Vesc_x']
            r_b = dict(r)
            def _mk(xt, vx):
                def g(tq): return vx(xt(tq))
                return g
            r_b['V_esc_t'] = _mk(r['x_of_t'], Vesc_x)
            r_b['V_esc_inf'] = cr['V_esc_inf']
            shots_b.append(r_b)
            V_esc_inf_b[i] = cr['V_esc_inf']

        lb_b = np.concatenate([[0.5, 0.0], V_esc_inf_b + 0.5])
        ub_b = np.concatenate([[10.0, 2.0], np.full(N_SHOTS, 200.0)])
        th = np.clip(th0.copy(), lb_b + 1e-6, ub_b - 1e-6)

        def f(theta, _s=shots_b):
            return residual_vector(theta, shots=_s, zls=z_launches)
        try:
            out = least_squares(f, th, bounds=(lb_b, ub_b),
                                diff_step=1e-3,
                                xtol=1e-8, ftol=1e-8, gtol=1e-8,
                                max_nfev=80)
            if out.status > 0:
                draws.append(out.x)
        except Exception:
            pass
        if (b + 1) % 10 == 0:
            print(f'  draw {b+1}/{B}  accepted={len(draws)}')
    return np.array(draws)

boot = refit_with_resampled_Vesc(B=50, seed=1, theta_start=theta_hat)
print(f'\nbootstrap: {boot.shape[0]} accepted draws of {50}\n')

q_lo = np.percentile(boot, 2.5, axis=0)
q_md = np.percentile(boot, 50,  axis=0)
q_hi = np.percentile(boot, 97.5, axis=0)
df_boot = pd.DataFrame({
    'param': param_names,
    'point_estimate': theta_hat,
    'boot_median': q_md,
    'boot_2.5%': q_lo,
    'boot_97.5%': q_hi,
    'hess_se': se,
})
print(df_boot.to_string(index=False, float_format=lambda x: f'{x:9.4f}'))


## 14. Joint $(C_{d,\rm eff},\,C_{AM})$ distribution

Scatter of bootstrap draws in $(C_{d,\rm eff},\,C_{AM})$ space with the
chapter Table 1.4 baseline overlaid for reference; histogram of the
bootstrap distribution of the well-determined combination
$R=(1+C_{AM})/C_{d,\rm eff}$.

In [ ]:
CHAPTER_BASELINE = (3.76, 0.69)  # (C_d_eff, C_AM) from chapter Table 1.4

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
ax.scatter(boot[:, 0], boot[:, 1], s=14, alpha=0.4, color='C0',
           label='bootstrap')
ax.plot(Cd_hat, CAM_hat, 'kx', ms=10, mew=2, label='point estimate')
ax.plot(*CHAPTER_BASELINE, 'r*', ms=12, label='chapter Table 1.4')
ax.set_xlabel(r'$C_{d,\rm eff}$'); ax.set_ylabel(r'$C_{AM}$')
ax.set_title(r'Joint $(C_{d,\rm eff},\,C_{AM})$ bootstrap'); ax.legend()

ax = axes[1]
R_boot = (1.0 + boot[:, 1]) / boot[:, 0]
ax.hist(R_boot, bins=20, color='C0', edgecolor='k', alpha=0.7)
ax.axvline(R_hat, color='k', lw=1.5, label=f'point: R={R_hat:.3f}')
ax.set_xlabel(r'$R = (1+C_{AM})/C_{d,\rm eff}$'); ax.set_ylabel('count')
ax.set_title(r'Universal combination'); ax.legend()
fig.tight_layout(); plt.show()

rho_boot = np.corrcoef(boot[:, 0], boot[:, 1])[0, 1]
print(f"bootstrap rho(C_d_eff, C_AM) = {rho_boot:+.3f}")
print(f"point R = {R_hat:.4f},  bootstrap median = {np.median(R_boot):.4f},"
      f"  IQR = [{np.percentile(R_boot,25):.4f}, "
      f"{np.percentile(R_boot,75):.4f}]")

## 15. Notes

### What to verify when reading the output

1. **Moment-matching vs L-BFGS-B optimum.** The closed-form moment-matching
   estimate should be within an order of magnitude of the L-BFGS-B optimum.
   Large discrepancy signals a problem with the early/late window choices,
   or that the universal-curve assumption is broken on this dataset.
2. **Bootstrap median vs point estimate.** Should agree to within the
   bootstrap SE. Systematic bias points to a small-$N$ event problem
   at high loading (the tail of $V_{\rm esc}(x)$ is undersampled).
3. **Hessian SE vs bootstrap IQR.** Should agree within a factor ~2; if they
   diverge by more, trust the bootstrap. The Hessian assumes a quadratic
   well at the optimum; bootstrap captures the genuine sampling distribution.

### Time-alignment caveat

Both clocks (trajectory $t$ and the implicit $t$ inside $V_{\rm esc}(t)$ via
$x_{\rm ring}(t)$) must share $t=0$. The model is sensitive to *when*
$V_{\rm esc}$ steps occur, not just the total — a constant time shift on
$V_{\rm esc}(t)$ trades off against $V_b^{(i)}(0)$ in a way the fit cannot
distinguish.

### Adding $U_p=200$ shots

The constant-$V_b$ pipeline excluded $U_p=200$ shots because $V_b(t)$ varied
appreciably inside the trajectory window, violating the constant-$V_b$
assumption. The ODE model handles $V_b(t)$ directly, so this exclusion no
longer applies — provided the §1.2 escape data for those conditions
($U_p=200$, $b=1$ and $b=3$, files `bubbles_06.csv` and `bubbles_05.csv`)
is well-enough sampled inside the trajectory window. Extend `TRAJ_SHOTS` and
the moment-matching block; nothing downstream needs to change.

### Connection to cross-check B

Where the constant-$V_b$ fit inferred a single $V_b$ that — by mass
conservation — equalled $V_b^{(i)}(\infty)$, the new fit reports the
*initial* loading $V_b^{(i)}(0)$. The cross-check $V_{\rm count}^{(i)} =
V_b^{(i)}(0) - V_b^{(i)}(\infty)$ becomes

$$
\frac{V_{\rm count}^{(i)}}{V_b^{(i)}(0)} \;\stackrel{?}{=}\; 1 - \phi^V_{\rm edge},
$$

which is just a rearrangement of eq. 1.39 — the same physical content, now
with the LHS expressed directly in terms of the fit's initial loading.